# SOV33 Sovereign Evaluation

**Benchmarks:** MMLU-Pro (200) · GSM8K (200) · AIME-2024 (30)

**Models:**
- `SOV33_small` → `Qwen/Qwen3-0.6B-Base`
- `SOV33_large` → `Qwen/Qwen3-30B-A3B`

Every inference is HMAC-SHA-256-signed (CSOAI SIGIL chain) and written to `submission.csv`.

Run all cells (`Run All`) — outputs:
1. `submission.csv` — `model_id, benchmark, score, sigil, timestamp`
2. `sigil_chain.jsonl` — per-question audit trail
3. `per_subject_<model>_<benchmark>.json` — per-subject accuracy

Toggle `RUN_SMALL` / `RUN_LARGE` in cell 2 to evaluate one model only.


In [ ]:
from __future__ import annotations

import csv, hashlib, hmac, json, os, random, re, sys, time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Iterable

# Toggle which models to evaluate (set both True for full run)
RUN_SMALL = True
RUN_LARGE = True

SOV33_PUBLIC_SECRET = b"SOV33-CSOAI-PUBLIC-CHAIN-v1"
KAGGLE_OUTPUT_DIR   = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./output")
KAGGLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_REGISTRY = {
    "SOV33_small": "Qwen/Qwen3-0.6B-Base",
    "SOV33_large": "Qwen/Qwen3-30B-A3B",
}

BENCHMARK_SIZES = {"MMLU-Pro": 200, "GSM8K": 200, "AIME-2024": 30}
RANDOM_SEED     = 20260713

try:
    import torch
    _HAS_TORCH = True
except Exception:
    _HAS_TORCH = False
    torch = None

DEVICE = ("cuda" if _HAS_TORCH and torch.cuda.is_available()
          else ("mps" if _HAS_TORCH and hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
                else "cpu"))
print(f"[init] device = {DEVICE}")


def emit_sigil(agent, model_id, benchmark, question_hash,
               prediction, ground_truth, verdict,
               secret=SOV33_PUBLIC_SECRET):
    payload = "|".join([agent, model_id, benchmark, question_hash,
                        str(prediction)[:120], str(ground_truth)[:120], verdict])
    digest = hmac.new(secret, payload.encode("utf-8"), hashlib.sha256).hexdigest()
    return f"H|{payload}|{digest}"


In [ ]:
def _seeded_subset(seq, k, seed=RANDOM_SEED):
    rng = random.Random(seed)
    idx = list(range(len(seq)))
    rng.shuffle(idx)
    return [seq[i] for i in sorted(idx[:k])]


def load_mmlu_pro(n=200):
    from datasets import load_dataset
    print("[data] loading MMLU-Pro …")
    ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test", trust_remote_code=True)
    by_subject = {}
    for row in ds:
        by_subject.setdefault(row["category"], []).append(dict(row))
    per_subject = max(1, n // max(1, len(by_subject)))
    sampled = []
    for subj, items in by_subject.items():
        sampled.extend(_seeded_subset(items, per_subject))
    if len(sampled) > n:
        sampled = _seeded_subset(sampled, n)
    elif len(sampled) < n:
        all_rows = [r for items in by_subject.values() for r in items]
        sampled.extend(_seeded_subset(all_rows, n)[: n - len(sampled)])
    print(f"[data] MMLU-Pro → {len(sampled)} questions across {len(by_subject)} subjects")
    return sampled


def load_gsm8k(n=200):
    from datasets import load_dataset
    print("[data] loading GSM8K …")
    ds = load_dataset("openai/gsm8k", "main", split="test", trust_remote_code=True)
    rows = _seeded_subset([dict(r) for r in ds], n)
    print(f"[data] GSM8K → {len(rows)} questions")
    return rows


def load_aime_2024():
    from datasets import load_dataset
    print("[data] loading AIME-2024 …")
    ds = load_dataset("Maxwell-Jia/AIME_2024", split="train", trust_remote_code=True)
    rows = [dict(r) for r in ds]
    print(f"[data] AIME-2024 → {len(rows)} questions")
    return rows


In [ ]:
MMLU_PRO_FEWSHOT = [
    {"q": "What is the capital of France?",  "choices": ["London", "Berlin", "Paris", "Madrid"],  "answer": "C"},
    {"q": "Solve 2x + 3 = 11",               "choices": ["x=2", "x=3", "x=4", "x=5"],            "answer": "C"},
    {"q": "Which gas do plants absorb?",     "choices": ["O2", "N2", "CO2", "H2"],               "answer": "C"},
    {"q": "Largest planet?",                 "choices": ["Earth", "Mars", "Jupiter", "Venus"],  "answer": "C"},
    {"q": "Author of \'1984\'?",              "choices": ["Huxley", "Orwell", "Tolkien", "Asimov"], "answer": "B"},
]

LETTERS = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"]


def build_mmlu_prompt(item):
    parts = ["The following are multiple choice questions (with answers).\n"]
    for fs in MMLU_PRO_FEWSHOT:
        choices_txt = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(fs["choices"]))
        parts.append(f"Question: {fs['q']}\n{choices_txt}\nAnswer: {fs['answer']}\n")
    choices_txt = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(item["options"]))
    parts.append(f"Question: {item['question']}\n{choices_txt}\nAnswer:")
    return "\n".join(parts)


GSM_FEWSHOT = [
    {"q": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
     "a": "Natalia sold 48/2 = 24 clips in May.\nNatalia sold 48+24 = 72 clips altogether in April and May.\n#### 72"},
    {"q": "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?",
     "a": "Weng earns 12/60 = $0.2 per minute.\nWorking 50 minutes, she earned 0.2 x 50 = $10.\n#### 10"},
    {"q": "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?",
     "a": "In the beginning, Betty has only 100 / 2 = $50.\nHer grandparents gave her 15 * 2 = $30.\nAdding the gifts, Betty now has 50 + 15 + 30 = $95.\nShe still needs 100 - 95 = $5.\n#### 5"},
    {"q": "Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?",
     "a": "Today she read 12*2 = 24 pages.\nSo far she read 12+24 = 36 pages.\nRemaining 120-36 = 84 pages.\nTomorrow she will read 84/2 = 42 pages.\n#### 42"},
    {"q": "James writes a 3-page letter to 2 different friends twice a week. How many pages does he write a year?",
     "a": "He writes each friend 3*2=6 pages a week.\nSo he writes 6*2=12 pages every week.\nThat means he writes 12*52=624 pages a year.\n#### 624"},
    {"q": "Mark has a garden with flowers. He planted plants of three different colors in it. Ten of them are yellow, and there are 80% more of those in purple. There are only 25% as many green flowers as there are yellow and purple flowers. How many flowers does Mark have in his garden?",
     "a": "There are 80/100*10=8 more purple flowers than yellow.\nSo there are 10+8=18 purple flowers.\nThere are 25/100*(10+18)=7 green flowers.\nTotal 10+18+7=35 flowers.\n#### 35"},
    {"q": "Albert is wondering how much pizza he can eat in one day. He buys 2 large pizzas and 2 small pizzas. A large pizza has 16 slices and a small pizza has 8 slices. If he eats it all, how many pieces does he eat that day?",
     "a": "Large pizza slices = 2*16=32.\nSmall pizza slices = 2*8=16.\nTotal = 32+16=48.\n#### 48"},
    {"q": "Ken created a care package to send to his brother, who was away at boarding school.  Ken placed a box on a scale, and then he poured into the box enough jelly beans to bring the weight to 2 pounds.  Then, he added enough brownies to cause the weight to triple.  Next, he added another 2 pounds of jelly beans.  And finally, he added enough gummy worms to double the weight once again.  What was the final weight of the box of goodies, in pounds?",
     "a": "Start with 2 lbs jelly beans.\nAdding brownies tripled the weight to 2*3=6 lbs.\nAdding 2 lbs jelly beans made it 6+2=8 lbs.\nAdding gummy worms doubled it to 8*2=16 lbs.\n#### 16"},
]


def build_gsm_prompt(item):
    parts = []
    for fs in GSM_FEWSHOT:
        parts.append(f"Question: {fs['q']}\nAnswer: {fs['a']}\n")
    parts.append(f"Question: {item['question']}\nAnswer:")
    return "\n".join(parts)


def build_aime_prompt(item):
    return ("You are an expert mathematician. Solve the following AIME problem. "
            "Give ONLY the integer answer 0-999.\n\n"
            f"Problem: {item['Problem']}\n\nAnswer:")


_LETTER_RE = re.compile(r"\b([A-J])\b")
_NUM_RE    = re.compile(r"-?\d+")


def extract_mmlu_answer(text):
    for line in text.splitlines():
        line = line.strip()
        if not line: continue
        m = _LETTER_RE.findall(line)
        if m: return m[-1]
    m = _LETTER_RE.findall(text)
    return m[-1] if m else "A"


def extract_gsm_answer(text):
    matches = _NUM_RE.findall(text.replace(",", ""))
    return matches[-1] if matches else ""


def extract_aime_answer(text):
    m = _NUM_RE.findall(text.replace(",", ""))
    return m[-1] if m else ""


In [ ]:
@dataclass
class HFModel:
    model_id: str
    short: str
    tokenizer: object = None
    model: object = None
    device: str = DEVICE
    n_params_b: float = 0.0

    def load(self):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        print(f"[model] loading {self.short} → {self.model_id} on {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id, trust_remote_code=True)
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        kw = dict(torch_dtype=dtype, trust_remote_code=True)
        if self.device in ("cuda", "mps"):
            kw["device_map"] = "auto"
        self.model = AutoModelForCausalLM.from_pretrained(self.model_id, **kw)
        if self.device == "cpu":
            self.model = self.model.to(self.device)
        self.model.eval()
        try:
            self.n_params_b = round(self.model.config.num_parameters() / 1e9, 2)
        except Exception:
            self.n_params_b = 0.0
        print(f"[model] {self.short} loaded ({self.n_params_b}B params)")
        return self

    def generate(self, prompt, max_new_tokens=256):
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs,
                                      max_new_tokens=max_new_tokens,
                                      do_sample=False, num_beams=1,
                                      pad_token_id=self.tokenizer.eos_token_id)
        return self.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


In [ ]:
@dataclass
class BenchResult:
    benchmark: str
    n: int
    correct: int
    per_subject: dict = field(default_factory=dict)
    latencies: list = field(default_factory=list)

    @property
    def accuracy(self): return self.correct / max(1, self.n)


def evaluate(model, benchmark, questions, *,
             build_prompt, extract_answer, ground_truth,
             subject_key=None, max_new_tokens=256, agent="SOV33-journal"):
    result = BenchResult(benchmark=benchmark, n=len(questions), correct=0)
    sigils = []
    print(f"\n[eval] {model.short} on {benchmark} ({len(questions)} questions)")
    for i, q in enumerate(questions, 1):
        prompt = build_prompt(q)
        gt     = ground_truth(q)
        t0     = time.time()
        try:
            raw = model.generate(prompt, max_new_tokens=max_new_tokens)
        except Exception as e:
            raw = ""
            print(f"[eval] ! error Q{i}: {e}")
        pred    = extract_answer(raw)
        correct = int(pred.strip() == gt.strip())
        lat_ms  = int((time.time() - t0) * 1000)
        q_hash  = hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]
        sigil   = emit_sigil(agent, model.short, benchmark, q_hash,
                             pred, gt, "OK" if correct else "MISS")
        result.correct += correct
        result.latencies.append(lat_ms)
        if subject_key and subject_key in q:
            sub = q[subject_key]
            result.per_subject.setdefault(sub, [0, 0])
            result.per_subject[sub][0] += correct
            result.per_subject[sub][1] += 1
        sigils.append({"ts": datetime.now(timezone.utc).isoformat(),
                       "model": model.short, "benchmark": benchmark,
                       "q_hash": q_hash, "pred": pred, "gt": gt,
                       "correct": bool(correct), "latency_ms": lat_ms,
                       "sigil": sigil})
        if i % 25 == 0 or i == len(questions):
            print(f"[eval]   … {i}/{len(questions)}  acc={result.correct/i:.3f}")
    return result, sigils


def ground_truth_mmlu(item): return LETTERS[int(item["answer_index"])]
def ground_truth_gsm(item):  return item["answer"].split("####")[-1].strip()
def ground_truth_aime(item): return str(item["Answer"])


def run_mmlu(model, n=BENCHMARK_SIZES["MMLU-Pro"]):
    items = load_mmlu_pro(n)
    return evaluate(model, "MMLU-Pro", items,
                    build_prompt=build_mmlu_prompt,
                    extract_answer=extract_mmlu_answer,
                    ground_truth=ground_truth_mmlu,
                    subject_key="category")


def run_gsm(model, n=BENCHMARK_SIZES["GSM8K"]):
    items = load_gsm8k(n)
    return evaluate(model, "GSM8K", items,
                    build_prompt=build_gsm_prompt,
                    extract_answer=extract_gsm_answer,
                    ground_truth=ground_truth_gsm,
                    max_new_tokens=320)


def run_aime(model):
    items = load_aime_2024()
    return evaluate(model, "AIME-2024", items,
                    build_prompt=build_aime_prompt,
                    extract_answer=extract_aime_answer,
                    ground_truth=ground_truth_aime,
                    max_new_tokens=512)


In [ ]:
def write_submission(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["model_id", "benchmark", "score", "sigil", "timestamp"])
        w.writeheader(); w.writerows(rows)
    print(f"[write] submission.csv → {path}  ({path.stat().st_size} bytes)")


def write_sigil_chain(rows, path):
    with path.open("w") as fh:
        for r in rows:
            fh.write(json.dumps(r) + "\n")
    print(f"[write] sigil_chain.jsonl → {path}  ({path.stat().st_size} bytes)")


# ===== MAIN =====
print("=" * 78)
print(" SOV33 Sovereign Evaluation – MMLU-Pro · GSM8K · AIME-2024")
print(f" Output dir : {KAGGLE_OUTPUT_DIR}")
print(f" Device     : {DEVICE}")
print(f" Seed       : {RANDOM_SEED}")
print(f" Sizes      : {BENCHMARK_SIZES}")
print("=" * 78)

models_to_run = []
if RUN_SMALL: models_to_run.append("SOV33_small")
if RUN_LARGE: models_to_run.append("SOV33_large")

submission_rows, sigil_chain, summary = [], [], {}
for short in models_to_run:
    if short not in MODEL_REGISTRY:
        print(f"[main] unknown model {short}"); continue
    model = HFModel(model_id=MODEL_REGISTRY[short], short=short).load()
    for runner in (run_mmlu, run_gsm, run_aime):
        res, sigs = runner(model)
        avg_sig = hmac.new(
            SOV33_PUBLIC_SECRET,
            f"{short}|{res.benchmark}|{res.accuracy:.6f}".encode(),
            hashlib.sha256,
        ).hexdigest()
        submission_rows.append({
            "model_id": short, "benchmark": res.benchmark,
            "score": f"{res.accuracy:.4f}", "sigil": avg_sig,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        sigil_chain.extend(sigs)
        summary.setdefault(short, {})[res.benchmark] = res.accuracy
        if res.per_subject:
            p = KAGGLE_OUTPUT_DIR / f"per_subject_{short}_{res.benchmark}.json"
            p.write_text(json.dumps(
                {k: {"correct": v[0], "total": v[1], "acc": v[0]/v[1]}
                 for k, v in sorted(res.per_subject.items())}, indent=2))
            print(f"[write] per-subject → {p}")
    if _HAS_TORCH and DEVICE == "cuda":
        import gc
        del model.model, model.tokenizer; gc.collect(); torch.cuda.empty_cache()

write_submission(submission_rows, KAGGLE_OUTPUT_DIR / "submission.csv")
write_sigil_chain(sigil_chain,   KAGGLE_OUTPUT_DIR / "sigil_chain.jsonl")

print("\n" + "=" * 78)
print(" FINAL SUMMARY")
print("=" * 78)
for short, scores in summary.items():
    print(f"{short}: MMLU {scores.get('MMLU-Pro', 0)*100:.1f}%, "
          f"GSM8K {scores.get('GSM8K', 0)*100:.1f}%, "
          f"AIME {scores.get('AIME-2024', 0)*100:.1f}%")
print("=" * 78)
